# Stage 2 — Musterhaushalte

One reform, worked end to end: a steeper withdrawal of earnings between 520 € and 1000 €
per month from Bürgergeld, swept over `bruttolohn_m` for a single adult, plotted as a
budget constraint and then differenced. Policy date 1 July 2023, the same as yesterday.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from gettsim import InputData, MainTarget, TTTargets, copy_environment, main
from gettsim.tt import (
    PiecewisePolynomialParam,
    TTSIMUnit,
    get_piecewise_parameters,
)
from gettsim_personas import grundsicherung_für_erwerbsfähige

POLICY_DATE = "2023-07-01"

## 1 · The two policy environments

The reform below is fenced off: replace what is between the markers with your own reform,
or leave the cell untouched and work with the demo reform.

In [ ]:
status_quo = main(
    main_target=MainTarget.policy_environment,
    policy_date_str=POLICY_DATE,
)

# ---------------------------- YOUR REFORM GOES HERE ----------------------------
# Paste the reform you built yesterday. If Thursday did not finish, leave this
# cell exactly as it is and work with the demo reform.

reform = copy_environment(status_quo)
reform["bürgergeld"]["parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg"] = (
    PiecewisePolynomialParam(
        value=get_piecewise_parameters(
            func_type="piecewise_linear",
            parameter_list=[
                {"interval": "(-inf, 0)", "intercept": 0, "slope": 0},
                {"interval": "[0, 100)", "slope": 1.0},
                {"interval": "[100, 520)", "slope": 0.2},
                {"interval": "[520, 1000)", "slope": 0.15},
                {"interval": "[1000, 1200)", "slope": 0.1},
                {"interval": "[1200, inf)", "slope": 0.0},
            ],
            leaf_name="parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg",
            xnp=np,
        ),
        input_unit=TTSIMUnit.EUR.PER_MONTH,
        output_unit=TTSIMUnit.EUR.PER_MONTH,
    )
)
# -------------------------------------------------------------------------------

## 2 · The persona

`SingleAdult` carries input data, targets and a policy date. Read the description before
using one: it says what has been assumed away. The other personas live in
`gettsim-personas/src/_gettsim_personas/de/`.

In [ ]:
persona = grundsicherung_für_erwerbsfähige.SingleAdult(policy_date_str=POLICY_DATE)
print(persona.description)

In [ ]:
persona.tt_targets_tree

## 3 · Varying earnings

`LinspaceGrid` sweeps `bruttolohn_m` for one or more `p_id`s. Everything else in the
household stays put.

In [ ]:
Single = grundsicherung_für_erwerbsfähige.SingleAdult

swept = Single(
    policy_date_str=POLICY_DATE,
    bruttolohn_m_linspace_grid=Single.LinspaceGrid(
        p0=Single.LinspaceRange(bottom=0, top=2_000),
        n_points=201,
    ),
)

swept.input_data_tree["einnahmen"]["bruttolohn_m"][:8]

Anything other than `bruttolohn_m` goes through `persona.upsert_input_data`. Positions in
the input arrays encode who is who, so check the structure before you overwrite anything.

## 4 · Running the grid under both environments

Nothing here is hidden in a helper — this is the loop, written out.

In [ ]:
TARGETS = {
    "einnahmen": {"bruttolohn_m": None},
    "bürgergeld": {"betrag_m_bg": None},
    "einkommensteuer": {"betrag_m_sn": None},
    "solidaritätszuschlag": {"betrag_m_sn": None},
    "sozialversicherung": {"beiträge_versicherter_m": None},
    "kindergeld": {"betrag_m": None},
    "kinderzuschlag": {"betrag_m_bg": None},
    "wohngeld": {"betrag_m_wthh": None},
}


def run(policy_environment, persona):
    """Run one persona under one policy environment; return a flat DataFrame."""
    result = main(
        main_target=MainTarget.results.df_with_nested_columns,
        policy_date=persona.policy_date,
        policy_environment=policy_environment,
        input_data=InputData.tree(persona.input_data_tree),
        tt_targets=TTTargets.tree(TARGETS),
        include_warn_nodes=False,
    )
    result.columns = ["__".join(c).strip("_") for c in result.columns]
    return result.reset_index()


before = run(status_quo, swept)
after = run(reform, swept)
before.head()

### Disposable income

GETTSIM has no single `disposable income` node, so we build it. The suffixes say at which
level each quantity is computed — `_bg` Bedarfsgemeinschaft, `_sn` jointly assessed
spouses, `_wthh` wohngeldrechtlicher Teilhaushalt, `_hh` household — and group values are
repeated on every member, so adding them up only works where the levels coincide. For a
single adult they all do; on real data they do not, which is notebook 03's problem.

In [ ]:
def disposable_income(df):
    """Monthly disposable income. Valid only where HH = BG = SN = WTHH."""
    return (
        df["einnahmen__bruttolohn_m"]
        - df["einkommensteuer__betrag_m_sn"]
        - df["solidaritätszuschlag__betrag_m_sn"]
        - df["sozialversicherung__beiträge_versicherter_m"]
        + df["bürgergeld__betrag_m_bg"]
        + df["kindergeld__betrag_m"]
        + df["kinderzuschlag__betrag_m_bg"]
        + df["wohngeld__betrag_m_wthh"]
    )


budget = pd.DataFrame(
    {
        "gross earnings": before["einnahmen__bruttolohn_m"],
        "status quo": disposable_income(before),
        "reform": disposable_income(after),
    }
)
budget.head()

## 5 · The budget constraint

In [ ]:
fig = go.Figure()
for name, colour in [("status quo", "#1f77b4"), ("reform", "#d62728")]:
    fig.add_scatter(
        x=budget["gross earnings"],
        y=budget[name],
        name=name,
        mode="lines",
        line={"color": colour, "width": 2},
    )
fig.add_scatter(
    x=budget["gross earnings"],
    y=budget["gross earnings"],
    name="45°",
    mode="lines",
    line={"dash": "dot", "color": "#999", "width": 1},
)
fig.update_layout(
    title="Budget constraint, single adult, 2023",
    xaxis_title="gross earnings, € per month",
    yaxis_title="disposable income, € per month",
    template="plotly_white",
    height=460,
)
fig

### The kinks

A kink is where the slope changes — where one more euro earned is retained at a
different rate. Differencing the budget constraint shows them directly.

In [ ]:
step = budget["gross earnings"].diff()
marginal = pd.DataFrame(
    {
        "gross earnings": budget["gross earnings"],
        "status quo": budget["status quo"].diff() / step,
        "reform": budget["reform"].diff() / step,
    }
)

fig = go.Figure()
for name, colour in [("status quo", "#1f77b4"), ("reform", "#d62728")]:
    fig.add_scatter(
        x=marginal["gross earnings"],
        y=marginal[name],
        name=name,
        mode="lines",
        line={"color": colour, "width": 2, "shape": "hv"},
    )
fig.update_layout(
    title="Marginal retention rate",
    xaxis_title="gross earnings, € per month",
    yaxis_title="Δ disposable income / Δ gross earnings",
    template="plotly_white",
    height=420,
)
fig